# Day 7: The Coffee Automaton

> Aaronson, Carroll, Ouellette (2014) — [arXiv:1405.6903](https://arxiv.org/abs/1405.6903)

This notebook walks through the paper's key experiments:
1. Build the coffee automaton (binary grid, random adjacent swaps)
2. Measure apparent complexity (gzip of coarse-grained state)
3. Show complexity rises then falls while entropy monotonically increases
4. Compare interacting vs non-interacting models
5. Replicate the adjusted coarse-graining (Section 6)

In [ ]:
import numpy as np
import gzip
import matplotlib.pyplot as plt
%matplotlib inline

from implementation import (
    CoffeeAutomaton, run_simulation, compare_models,
    coarse_grain, threshold_array, row_majority_adjust,
    measure_entropy, measure_complexity, gzip_size, grid_to_bytes,
    DEFAULT_GRAIN
)
print('Imports OK')

## 1. The Coffee Automaton Model

Section 3.1: N x N binary grid. Top half = cream (1), bottom half = coffee (0).
At each step, pick a random adjacent pair that differs, and swap them.

For speed the repo applies a *batch* of disjoint swaps per step (one step here ≈ 0.07·N² swaps once mixing is under way);
`ca.swaps` counts the individual swaps, which is the paper's time unit.

In [ ]:
# Create a 50x50 interacting automaton
ca = CoffeeAutomaton(grid_size=50, model='interacting', seed=42)

# Visualize initial state
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
states = [0, 1000, 2500, 10000]   # batches; the 50x50 cup is mixed by ~10,000

for i, t in enumerate(states):
    if t > ca.time:
        ca.step(t - ca.time)
    axes[i].imshow(ca.grid, cmap='gray_r', vmin=0, vmax=1)
    axes[i].set_title(f't = {t} batches ({ca.swaps:.2g} swaps)')
    axes[i].axis('off')

plt.suptitle('Coffee Automaton: Interacting Model (50x50)')
plt.tight_layout()
plt.show()

## 2. Measuring Apparent Complexity

The paper's key measurement (Section 5):
- **Entropy**: gzip(fine-grained state) — proxy for K(x)
- **Complexity**: gzip(coarse-grained state) — proxy for K(f(x))

Coarse-graining (Sec. 5.1): replace every cell by the average of the g x g window *centred* on it (the result is still N x N; windows at the edge average their in-bounds cells), threshold into buckets, compress. We use g = 7 for every grid size (`DEFAULT_GRAIN`); the paper chose g "experimentally" and does not report it.

In [ ]:
# Run full simulation and collect measurements
print('Running simulation (50x50, 4 N^2 = 10000 batches)...')
results = run_simulation(
    grid_size=50, num_snapshots=80,
    model='interacting', seed=42
)

# Plot entropy and complexity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results['times'], results['entropy'], 'b-')
ax1.set_xlabel('Time (batches of swaps)')
ax1.set_ylabel('Entropy (gzip bytes)')
ax1.set_title('Entropy: Monotonically Increasing')

ax2.plot(results['times'], results['complexity'], 'r-')
peak_idx = np.argmax(results['complexity'])
ax2.axvline(results['times'][peak_idx], color='gray', ls='--', alpha=0.5)
ax2.set_xlabel('Time (batches of swaps)')
ax2.set_ylabel('Complexity (gzip bytes)')
ax2.set_title('Apparent Complexity: Rises Then Falls')

plt.tight_layout()
plt.show()
print(f'Complexity {results["complexity"][0]} -> peak {results["complexity"][peak_idx]} -> '
      f'final {results["complexity"][-1]} bytes; peak at t = {results["times"][peak_idx]} batches '
      f'= {results["paper_time"][peak_idx]:.2g} swaps')

## 3. Coarse-Graining Visualization

Compare fine-grained and coarse-grained states at the complexity peak.
This is analogous to Figures 3 and 11 in the paper.

In [ ]:
# Show fine-grained and coarse-grained at different times
if 'grids' in results:
    indices = [0, peak_idx, len(results['grids'])-1]
    labels = ['t=0 (start)', f't={results["times"][peak_idx]} (peak)', f't={results["times"][-1]} (end)']
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for col, (idx, label) in enumerate(zip(indices, labels)):
        grid = results['grids'][idx]
        axes[0, col].imshow(grid, cmap='gray_r', vmin=0, vmax=1)
        axes[0, col].set_title(label)
        axes[0, col].axis('off')
        
        coarse = coarse_grain(grid, grain_size=DEFAULT_GRAIN)   # centred window: same 50 x 50 size
        discrete = threshold_array(coarse, 7)
        discrete = row_majority_adjust(discrete)
        axes[1, col].imshow(discrete, cmap='gray_r', vmin=0, vmax=6)
        axes[1, col].axis('off')
    
    axes[0, 0].set_ylabel('Fine-grained', fontsize=11)
    axes[1, 0].set_ylabel('Coarse-grained', fontsize=11)
    plt.suptitle('Fine-Grained vs Coarse-Grained States')
    plt.tight_layout()
    plt.show()

## 4. Interacting vs Non-Interacting

What the paper reports: with 3 buckets (Sec. 5.2) *both* models rise and fall; with the adjusted
7-bucket method (Sec. 6.2) the interacting curve keeps its shape while the non-interacting one is
"flattened to a lower value". The Appendix proves the non-interacting coarse-grained state stays simple,
but only for grains much larger than G·sqrt(3 ln 2n²) (about 35 cells at n = 50, G = 7).

The two models count time differently (a batch of swaps vs. every particle moving once), so each gets its
own run length and its own panel. The non-interacting state is the particle *count* per cell.

In [ ]:
# Compare both models with the same grain and thresholds (7 buckets + adjustment)
inter, non_inter = compare_models(grid_size=50, num_snapshots=60, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
axes[0].plot(inter['times'], inter['complexity'], 'r-')
axes[0].set_xlabel('Time (batches of swaps)')
axes[0].set_ylabel('Complexity (bytes)')
axes[0].set_title('Interacting')
axes[1].plot(non_inter['times'], non_inter['complexity'], 'b-')
axes[1].set_xlabel('Time (steps: every particle moves)')
axes[1].set_title('Non-interacting')
plt.suptitle('Apparent complexity, 7x7 centred window, 7 buckets + row-majority adjustment')
plt.tight_layout()
plt.show()

for name, r in (('interacting', inter), ('non-interacting', non_inter)):
    c = r['complexity']
    print(f'{name:16s} start {c[0]}, peak {c.max()}, end {c[-1]} bytes')
# At this size the non-interacting peak is NOT the lower one (see README: grain far below the
# Appendix's bound, and particle counts fluctuate more than 0/1 cells).

## 5. Section 5 vs Section 6: Thresholding Methods

Section 5 thresholds into 3 buckets. Section 6 argues that cells sitting on a bucket border flicker between
buckets and add spurious complexity, and counters it with 7 buckets plus a row-majority adjustment
(a cell within one bucket of its row's majority is set to the majority).

In [ ]:
# Compare 3-bucket (Section 5) vs 7-bucket adjusted (Section 6)
results_s5 = run_simulation(
    grid_size=50, num_snapshots=80,
    model='interacting', num_buckets=3, adjust=False, seed=42
)
results_s6 = run_simulation(
    grid_size=50, num_snapshots=80,
    model='interacting', num_buckets=7, adjust=True, seed=42
)

plt.figure(figsize=(10, 5))
plt.plot(results_s5['times'], results_s5['complexity'], 'r-',
         label='3-bucket, no adjustment (Section 5)', alpha=0.7)
plt.plot(results_s6['times'], results_s6['complexity'], 'b-',
         label='7-bucket, adjusted (Section 6)', alpha=0.7)
plt.xlabel('Time (batches of swaps)')
plt.ylabel('Apparent complexity (bytes)')
plt.title('Comparing Coarse-Graining Methods')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

What the runs above show (50x50, paper's centred-window coarse-graining, grain 7):

1. Apparent complexity (gzip of the coarse-grained state) rises then falls, under both the 3-bucket and the adjusted 7-bucket metric
2. Entropy (gzip of the fine-grained state) rises to its maximum and stays there
3. The adjusted metric does *not* single out the interacting model at this size: both models rise and fall, and the
   non-interacting peak is the higher one. The paper's Sec. 6.2 contrast is not reproduced here (README explains why)
4. `python train_minimal.py --scaling` shows max complexity growing with n, as in the paper's Fig. 7

The open problem: prove analytically that the interacting model must become complex.